# AlphaFold 2 Structure Prediction with `foldrun-shim` (FoldRun Cloud Gateway)

> **Adapted from Google DeepMind's canonical [`notebooks/AlphaFold.ipynb`](https://github.com/google-deepmind/alphafold/blob/v2.3.2/notebooks/AlphaFold.ipynb) (`google-deepmind/alphafold` v2.3.2).**
>
> Upstream `google-deepmind/alphafold` retired the public `AlphaFold.ipynb` Colab notebook (`"SERVICE UNAVAILABLE — This notebook is no longer supported"`) because standard Colab runtimes lack the **2.5 TB disk** required for AlphaFold 2's genetic and structural template databases, and streaming 300 database chunks over HTTP into a 9 GB RAM disk with `num_templates=0` frequently timed out on single T4 GPUs.
>
> This notebook preserves the familiar DeepMind AlphaFold 2 Colab workflow (sequence validation, monomer & multimer support, MSA coverage plots, pLDDT & PAE confidence plots, and interactive 3D `py3Dmol` visualization) while routing execution through **`foldrun-shim`** to FoldRun's **2.5 TB SSD-backed Cloud Filestore NFS (`/mnt/nfs/foldrun`)** and **auto-scaling Vertex AI NVIDIA A100 GPU Pipelines**.


---

## Original Notebook Reference, Credits & Citations

- **Original Public Notebook**: [Google DeepMind `notebooks/AlphaFold.ipynb` (v2.3.2)](https://github.com/google-deepmind/alphafold/blob/v2.3.2/notebooks/AlphaFold.ipynb) ([Open Original in Colab](https://colab.research.google.com/github/deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb))
- **Upstream Repository**: [`google-deepmind/alphafold`](https://github.com/google-deepmind/alphafold)
- **Authors & Credits**: John Jumper, Richard Evans, Alexander Pritzel, Tim Green, Michael Figurnov, Olaf Ronneberger, Kathryn Tunyasuvunakool, Russ Bates, Augustin Žídek, Anna Potapenko, Alex Bridgland, Clemens Meyer, Simon A. A. Kohl, Andrew J. Ballard, Andrew Cowie, Bernardino Romera-Paredes, Stanislav Nikolov, Rishub Jain, Jonas Adler, Trevor Back, Stig Petersen, David Reiman, Ellen Clancy, Michal Zielinski, Martin Steinegger, Michalina Pacholska, Tamas Berghammer, Sebastian Bodenstein, David Silver, Oriol Vinyals, Andrew W. Senior, Koray Kavukcuoglu, Pushmeet Kohli, and Demis Hassabis (**Google DeepMind**).
- **License**: Apache License 2.0 (source code) / CC-BY 4.0 (AlphaFold 2 model parameters).
- **Primary Citations**:
  1. Jumper, J. *et al.* "Highly accurate protein structure prediction with AlphaFold." *Nature* 596, 583–589 (2021). [doi:10.1038/s41586-021-03819-2](https://doi.org/10.1038/s41586-021-03819-2)
  2. Evans, R. *et al.* "Protein complex prediction with AlphaFold-Multimer." *bioRxiv* (2021). [doi:10.1101/2021.10.04.463034](https://doi.org/10.1101/2021.10.04.463034)

---

## What Changed: Original DeepMind AlphaFold Colab vs. `foldrun-shim`

| Stage | Original DeepMind `AlphaFold.ipynb` (v2.3.2) | Modified `AlphaFold2_FoldRun_Shim.ipynb` (`foldrun-shim`) |
| :--- | :--- | :--- |
| **1. Environment Setup** | ~20–30 min: Uninstall TensorFlow, `apt install hmmer`, wipe `/opt/conda`, install Miniconda + Python 3.9 + `openmm=7.5.1` + `pdbfixer`, patch OpenMM, clone `alphafold`, download 5.3 GB weights, mount 9 GB `tmpfs` `/tmp/ramdisk`. | **~5 seconds**: `pip install -e sdk/foldrun-shim py3Dmol matplotlib numpy` + set `FOLDRUN_GATEWAY_URL`. Zero Conda, HMMER, OpenMM, or JAX compilation on the client. |
| **2. Local Hardware Requirement** | **Hard GPU requirement**: Raises `RuntimeError('Colab CPU runtime not supported')` without a local GPU; limited to single Colab T4/L4 GPU VRAM. | **Zero local GPU or disk required**: Runs from any CPU-only laptop, free CPU Colab, Colab Enterprise, or Vertex AI Workbench. |
| **3. Genetic Databases & Templates** | Streamed **300 HTTP chunks** (`uniref90`, `smallbfd`, `mgnify`, `uniprot`) into `/tmp/ramdisk` on every run; **no templates (`num_templates=0`)**; **no `full_dbs`** (1.8 TB BFD + 206 GB UniRef30 impossible in Colab). | Direct high-throughput access to FoldRun's pre-provisioned **2.5 TB `BASIC_SSD` Cloud Filestore NFS (`/mnt/nfs/foldrun`)**: supports **both `--db_preset=full_dbs` and `reduced_dbs`**, **full PDB70 / PDB SeqRes + PDB mmCIF templates**, and **GPU-accelerated `mmseqs2`** or CPU `jackhmmer`/`hhblits`. |
| **4. Feature Caching & Deduplication** | None — re-streams all 300 HTTP chunks from scratch if you re-run or change model parameters. | **Two-level caching**: (1) NFS feature cache (`/mnt/nfs/foldrun/af2_features_cache/<preset>_<sha256>/features.pkl`) skips MSA search on repeat sequences; (2) Gateway job deduplication via `input_sha256`. |
| **5. Model Inference & Relaxation** | Runs 5 monomer models (or 5 × $N$ multimer predictions) + AMBER relaxation **sequentially** in a single-GPU loop. | Fans out all 5 models (up to 25 multimer predictions/model) **in parallel across multiple NVIDIA A100 GPUs** (`a2-highgpu-1g` 40 GB or `a2-ultragpu-1g` 80 GB) via KFP `dsl.ParallelFor` and DWS `FLEX_START`. |
| **6. Disconnect Resilience** | Browser tab sleep or Colab timeout kills the process and loses 100% of progress (`"Your Colab runtime may have died... restart and run from the first cell!"`). | Persists `<output_dir>/.foldrun_job.json` upon `202 Accepted`. Re-running the cell **automatically re-attaches** to the running Vertex AI `PipelineJob` without restarting or double-paying for A100 GPUs. |
| **7. Output Format & 3D Viewer** | Writes a single `selected_prediction.pdb` and `predicted_aligned_error.json`. | Materializes the complete native AlphaFold 2 directory (`<stem>/ranked_0..4.pdb`, `ranking_debug.json`, `relaxed_model_*.pdb`, `unrelaxed_model_*.pdb`, `features.pkl`, `msas/`) locally **plus** triggers `foldrun-analysis-job` and returns a hosted **3D FoldRun Viewer URL** (`job.viewer_url`). |


## 1. Prerequisites, Authentication & Installing `foldrun-shim` from GCS

During FoldRun deployment (`./deploy-all.sh` or `./deploy-all.sh --steps build --build-target sdk`), Cloud Build packages `foldrun-shim` into a pure-Python wheel and uploads both the wheel and all 7 modified public notebooks to your project's FoldRun GCS bucket under `gs://<PROJECT_ID>-foldrun-data/sdk/`:
- **Pre-Built Wheel**: `gs://<PROJECT_ID>-foldrun-data/sdk/latest_build` (and dated incremental builds)
- **Modified Notebooks**: `gs://<PROJECT_ID>-foldrun-data/sdk/notebooks/<model>/*.ipynb`

### Step-by-Step Setup (Executed in the Next Cell)

1. **Step 1 — Authenticate to Google Cloud First**:
   - **Google Colab**: Calls `from google.colab import auth; auth.authenticate_user()` so `gcloud storage cp` and `gcloud auth print-identity-token` have valid credentials before downloading the wheel.
   - **Colab Enterprise / Vertex AI Workbench**: Authenticated automatically via the runtime's attached service account and the Google Cloud Metadata Server.
   - **Local Jupyter Kernel**: Authenticate beforehand with `gcloud auth login`.
2. **Step 2 — Automatic Configuration Loading (`.env`)**:
   - FoldRun's deployment script (`./deploy-all.sh`) generates local `.env` files in `sdk/foldrun-shim/notebooks/` and uploads `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`.
   - The cell below automatically loads `.env` (searching `./.env`, `../.env`, `/content/.env`, or downloading from `gs://${FOLDRUN_GCS_BUCKET}/sdk/.env`), populating `GCP_PROJECT_ID`, `GCP_REGION`, `FOLDRUN_GCS_BUCKET`, `FOLDRUN_GATEWAY_URL`, and `FOLDRUN_VIEWER_URL`.
3. **Step 3 — Download & Install `foldrun-shim` from the FoldRun GCS Bucket**:
   Requires `roles/storage.objectViewer` on `gs://<PROJECT_ID>-foldrun-data` (ideally scoped to the `sdk/` prefix via an IAM condition):
   ```bash
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/latest_build /tmp/
   WHL=$(cat /tmp/latest_build)
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/${WHL} /tmp/
   pip install -q /tmp/${WHL}
   ```
4. **Step 4 — Configure `FOLDRUN_GATEWAY_URL` & OIDC Gateway Auth (`roles/run.invoker`)**:
   `foldrun_shim.auth.fetch_gateway_headers()` automatically attaches `Authorization: Bearer <OIDC_ID_TOKEN>` to every gateway request via:
   1. `FOLDRUN_ID_TOKEN` (if explicitly set),
   2. `FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT` (`gcloud auth print-identity-token --impersonate-service-account=...`),
   3. Google Cloud Metadata Server (`google.oauth2.id_token.fetch_id_token` on Vertex AI Workbench / Colab Enterprise),
   4. `gcloud auth print-identity-token` fallback (Colab / local session).

In [ ]:
# @title 1. Authenticate to Google Cloud & Install `foldrun-shim` from GCS
# @markdown Authenticate first, query `latest_build` from `gs://<PROJECT_ID>-foldrun-data/sdk/`, download the newest wheel, and configure `FOLDRUN_GATEWAY_URL`. **No local GPU is required.**

import os
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def _ensure_scripts_on_path() -> None:
    """Ensure pip console scripts (including --user installs) are discoverable on PATH."""
    script_dirs: list[str] = []
    for scheme in (None, f"{os.name}_user"):
        try:
            p = (
                sysconfig.get_path("scripts")
                if scheme is None
                else sysconfig.get_path("scripts", scheme)
            )
            if p:
                script_dirs.append(p)
        except Exception:
            pass
    script_dirs.append(str(Path(sys.executable).parent))
    current_parts = os.environ.get("PATH", "").split(os.pathsep) if os.environ.get("PATH") else []
    for d in reversed(script_dirs):
        if d and d not in current_parts:
            current_parts.insert(0, d)
    os.environ["PATH"] = os.pathsep.join(current_parts)


def _is_placeholder_value(val: str) -> bool:
    lowered = (val or "").strip().lower()
    return not lowered or any(
        tok in lowered
        for tok in (
            "your-project-foldrun-data",
            "your-project",
            "your_project",
            "example",
            "<",
        )
    )


def _is_placeholder_bucket(bucket: str) -> bool:
    clean = (bucket or "").strip().removeprefix("gs://").rstrip("/")
    return _is_placeholder_value(clean)


_ensure_scripts_on_path()


def _load_foldrun_dotenv(custom_path: Path | None = None) -> Path | None:
    """Load key=value pairs from a local .env file generated by deploy-all.sh."""
    if custom_path is not None:
        candidates = [custom_path]
    else:
        candidates = []
        if os.environ.get("FOLDRUN_ENV_FILE"):
            candidates.append(Path(os.environ["FOLDRUN_ENV_FILE"]))
        candidates.extend(
            [Path(".env"), Path("../.env"), Path("../../.env"), Path("/content/.env")]
        )
    loaded_from: Path | None = None
    for path in candidates:
        if not path.is_file():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = re.sub(r"^export(?=\s)\s*", "", k.strip())
            v = v.strip().strip("'").strip('"')
            if k and v:
                existing = os.environ.get(k, "")
                if not existing or _is_placeholder_value(existing):
                    os.environ[k] = v
        if loaded_from is None:
            loaded_from = path.resolve()
        if custom_path is not None or not _is_placeholder_value(
            os.environ.get("FOLDRUN_GATEWAY_URL", "")
        ):
            return path.resolve()
    return loaded_from


_dotenv_path = _load_foldrun_dotenv()
if _dotenv_path:
    print(f"Loaded FoldRun configuration from {_dotenv_path}")

FOLDRUN_GCS_BUCKET = "your-project-foldrun-data"  # @param {type:"string"}
FOLDRUN_GATEWAY_URL = "https://foldrun-gateway-example-uc.a.run.app"  # @param {type:"string"}
FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT = ""  # @param {type:"string"}

if not _is_placeholder_bucket(FOLDRUN_GCS_BUCKET):
    os.environ["FOLDRUN_GCS_BUCKET"] = FOLDRUN_GCS_BUCKET.strip()
else:
    os.environ.setdefault("FOLDRUN_GCS_BUCKET", "your-project-foldrun-data")

if not _is_placeholder_value(FOLDRUN_GATEWAY_URL):
    os.environ["FOLDRUN_GATEWAY_URL"] = FOLDRUN_GATEWAY_URL.strip()
else:
    os.environ.setdefault("FOLDRUN_GATEWAY_URL", "https://foldrun-gateway-example-uc.a.run.app")

if FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT.strip() and not _is_placeholder_value(
    FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT
):
    os.environ["FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT"] = FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT.strip()

# 1. Authenticate to Google Cloud first (required in standard Google Colab before accessing GCS)
if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()


def _install_foldrun_shim_from_gcs(gcs_bucket: str) -> None:
    """Download latest foldrun_shim wheel via latest_build from gs://<bucket>/sdk/ and install via pip."""
    clean_bucket = (gcs_bucket or "").strip().removeprefix("gs://").rstrip("/")
    bucket_configured = not _is_placeholder_bucket(clean_bucket)

    if bucket_configured and _is_placeholder_value(os.environ.get("FOLDRUN_GATEWAY_URL", "")):
        try:
            env_dst = Path("/tmp/foldrun_sdk.env")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/.env",
                    str(env_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and env_dst.is_file():
                if _load_foldrun_dotenv(env_dst):
                    print(f"Loaded remote FoldRun configuration from gs://{clean_bucket}/sdk/.env")
        except Exception as exc:
            print(f"Notice: could not load gs://{clean_bucket}/sdk/.env: {exc}")

    whl_filename = "foldrun_shim-0.1.0-py3-none-any.whl"
    if bucket_configured:
        try:
            latest_build_dst = Path("/tmp/foldrun_latest_build")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/latest_build",
                    str(latest_build_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and latest_build_dst.is_file():
                candidate = latest_build_dst.read_text(encoding="utf-8").strip()
                if candidate.endswith(".whl"):
                    whl_filename = candidate
                    print(f"Detected latest FoldRun SDK build: {whl_filename}")
        except Exception as exc:
            print(f"Notice: could not read gs://{clean_bucket}/sdk/latest_build: {exc}")

    whl_path = Path(f"/tmp/{whl_filename}")
    marker_path = Path(f"/tmp/{whl_filename}.installed")
    if marker_path.exists():
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            return
        except ImportError:
            pass

    if not bucket_configured:
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            print(
                "Notice: FOLDRUN_GCS_BUCKET is unconfigured; using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"foldrun_shim is not installed. Set FOLDRUN_GCS_BUCKET='<PROJECT_ID>-foldrun-data' "
                f"so this cell can download gs://<PROJECT_ID>-foldrun-data/sdk/{whl_filename}."
            )

    gcs_uri = f"gs://{clean_bucket}/sdk/{whl_filename}"
    print(f"Downloading {whl_filename} from {gcs_uri}...")
    cp_res = subprocess.run(
        ["gcloud", "storage", "cp", gcs_uri, str(whl_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if cp_res.returncode != 0:
        err_line = (
            (cp_res.stderr or cp_res.stdout or "gcloud storage cp failed").strip().splitlines()[-1]
        )
        try:
            import foldrun_shim  # noqa: F401

            _ensure_scripts_on_path()
            print(
                f"Warning: could not download {gcs_uri} ({err_line}); using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"Failed to download {gcs_uri} ({err_line}) and foldrun_shim is not installed locally."
            )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(whl_path)],
        check=True,
    )
    _ensure_scripts_on_path()
    marker_path.write_text("installed", encoding="utf-8")
    print(f"Installed {whl_filename} successfully.")


_install_foldrun_shim_from_gcs(os.environ.get("FOLDRUN_GCS_BUCKET", FOLDRUN_GCS_BUCKET))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "py3Dmol", "matplotlib", "numpy"],
    check=False,
)

import foldrun_shim
from foldrun_shim import alphafold2

print("foldrun-shim loaded successfully!")
print(f"foldrun-shim version: {getattr(foldrun_shim, '__version__', 'unknown')}")
print(
    f"run_alphafold.py CLI in PATH: {shutil.which('run_alphafold.py') or 'available via python -m / foldrun_shim.cli'}"
)
print(f"Configured FOLDRUN_GATEWAY_URL: {os.environ.get('FOLDRUN_GATEWAY_URL', '<not set yet>')}")

In [ ]:
# @title 1B. (Optional) Check Live GPU Availability & Configure On-Demand vs. DWS Scheduling
# @markdown Thin-permission notebook users (who only have `roles/run.invoker` on the FoldRun Gateway)
# @markdown can inspect live GPU availability, active/queued jobs, and regional quotas directly from Python
# @markdown without needing GCP project-level GCE or ServiceUsage IAM permissions.
# @markdown
# @markdown **CLI Equivalents:**
# @markdown - `!foldrun gpus` (inspect live GPU availability and queue depths)
# @markdown - `!foldrun status <job_id>` (inspect status of an active or queued job)
# @markdown - `!foldrun cancel <job_id>` (cancel a queued or running job)
# @markdown - GCP Project Admins only: `!gcloud alpha services quota list --service=aiplatform.googleapis.com --consumer=projects/$GCP_PROJECT_ID`

import os

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. Query live GPU availability and queue status via FoldRun Gateway
try:
    import foldrun_shim

    gpu_status = foldrun_shim.check_gpus()
    display(gpu_status)
except Exception as exc:
    print(f"Notice: Could not query live GPU status via FoldRun Gateway: {exc}")
    print(
        "You can still proceed with prediction using default scheduling or environment variables below."
    )

# 2. Scheduling & Hardware Configuration (Optional Overrides)
# Uncomment or adjust any of the following environment variables to control hardware & scheduling:

# GPU Architecture: "auto" (default), "L4" (fast & cost-effective), "A100" (40GB), or "A100_80GB"
# os.environ["FOLDRUN_GPU_TYPE"] = "L4"

# Scheduling Strategy:
# - "true"  -> Dynamic Workload Scheduler (DWS) FLEX_START (saves 45-68% cost; may queue if regional DWS capacity is constrained)
# - "false" -> Immediate On-Demand STANDARD GPUs (provisions immediately without DWS queueing)
# os.environ["FOLDRUN_ENABLE_FLEX_START"] = "false"

# Optional Queue Timeout: Automatically cancel the job if queued in DWS longer than N seconds
# os.environ["FOLDRUN_QUEUE_TIMEOUT_SEC"] = "300"

---
## Part A — Original DeepMind `AlphaFold.ipynb` Form & Visualization Workflow

This part provides the canonical interactive Colab form from Google DeepMind's official [`AlphaFold.ipynb`](https://github.com/google-deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb). Enter one or more amino acid sequences below for monomer or multimer prediction.


## 2. Enter the Amino Acid Sequence(s) to Fold (Monomer & Multimer)

Paste the amino acid sequence(s) of your protein below:
- **Single sequence (`MONOMER`)**: Runs AlphaFold 2 monomer (`monomer`, `monomer_ptm`, or `monomer_casp14`), or `multimer` if `use_multimer_model_for_monomers` is checked.
- **Multiple sequences (`MULTIMER`)**: Runs AlphaFold-Multimer across all chains (`A`, `B`, `C`, ...).

> [!TIP]
> **Automatic Vertex AI A100 Hardware Tiering (`gpu_type="auto"`)**
> When submitted to `foldrun-gateway`, FoldRun inspects total residue length and chain count (`AF2Tool._recommend_gpu`) to provision the optimal GPU tier automatically via Dynamic Workload Scheduler (`FLEX_START`):
> - **Monomer $\le 1500$ residues** or **Multimer $< 1000$ residues** $\rightarrow$ **NVIDIA A100 40 GB** (`a2-highgpu-1g`)
> - **Monomer $> 1500$ residues** or **Multimer $\ge 1000$ residues** $\rightarrow$ **NVIDIA A100 80 GB** (`a2-ultragpu-1g` for prediction; `a2-highgpu-1g` A100 40 GB for AMBER relaxation)

In [ ]:
# @title 2. Enter the amino acid sequence(s) to fold ⬇️
# @markdown Enter one sequence for Monomer prediction, or multiple sequences for AlphaFold-Multimer:

import enum


@enum.unique
class ModelType(enum.Enum):
    MONOMER = 0
    MULTIMER = 1


target_name = "T1024_monomer"  # @param {type:"string"}
sequence_1 = "MAAHKGAEHHHKAAEHHEQAAKHHHAAAEHHEKGEHEQAAHHADTAYAHHKHAEEHAAQAAKHDAEHHAPKPH"  # @param {type:"string"}
sequence_2 = ""  # @param {type:"string"}
sequence_3 = ""  # @param {type:"string"}
sequence_4 = ""  # @param {type:"string"}
sequence_5 = ""  # @param {type:"string"}
sequence_6 = ""  # @param {type:"string"}
sequence_7 = ""  # @param {type:"string"}
sequence_8 = ""  # @param {type:"string"}

use_multimer_model_for_monomers = False  # @param {type:"boolean"}

input_sequences = (
    sequence_1,
    sequence_2,
    sequence_3,
    sequence_4,
    sequence_5,
    sequence_6,
    sequence_7,
    sequence_8,
)

MIN_PER_SEQUENCE_LENGTH = 16
MAX_PER_SEQUENCE_LENGTH = 4000
MAX_MONOMER_MODEL_LENGTH = 2500
MAX_LENGTH = 4000
MAX_VALIDATED_LENGTH = 3000
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")


def clean_and_validate_input_sequences(
    raw_sequences: tuple[str, ...],
    min_length: int,
    max_length: int,
) -> list[str]:
    """Standalone equivalent of alphafold.notebooks.notebook_utils.clean_and_validate_input_sequences."""
    cleaned_list: list[str] = []
    for seq in raw_sequences:
        clean_seq = seq.translate(str.maketrans("", "", " \n\t")).upper()
        if not clean_seq:
            continue
        invalid_chars = set(clean_seq) - STANDARD_AA
        if invalid_chars:
            raise ValueError(
                f"Input sequence contains non-amino acid letters: {sorted(invalid_chars)}. "
                "AlphaFold only supports the 20 standard amino acids."
            )
        if len(clean_seq) < min_length:
            raise ValueError(
                f"Input sequence is too short: {len(clean_seq)} amino acids (minimum is {min_length})."
            )
        if len(clean_seq) > max_length:
            raise ValueError(
                f"Input sequence is too long: {len(clean_seq)} amino acids (maximum is {max_length})."
            )
        cleaned_list.append(clean_seq)
    if not cleaned_list:
        raise ValueError(
            "No input amino acid sequence provided; please provide at least one sequence."
        )
    return cleaned_list


sequences = clean_and_validate_input_sequences(
    raw_sequences=input_sequences,
    min_length=MIN_PER_SEQUENCE_LENGTH,
    max_length=MAX_PER_SEQUENCE_LENGTH,
)

if len(sequences) == 1:
    if use_multimer_model_for_monomers:
        print("Using the multimer model for single-chain, as requested.")
        model_type_to_use = ModelType.MULTIMER
    else:
        print("Using the single-chain (monomer) model.")
        model_type_to_use = ModelType.MONOMER
else:
    print(f"Using the multimer model with {len(sequences)} chains.")
    model_type_to_use = ModelType.MULTIMER

total_sequence_length = sum(len(s) for s in sequences)
if total_sequence_length > MAX_LENGTH:
    raise ValueError(
        f"Total sequence length ({total_sequence_length}) exceeds maximum ({MAX_LENGTH})."
    )
if model_type_to_use == ModelType.MONOMER and len(sequences[0]) > MAX_MONOMER_MODEL_LENGTH:
    raise ValueError(
        f"Monomer sequence length ({len(sequences[0])}) exceeds {MAX_MONOMER_MODEL_LENGTH}. "
        "Enable use_multimer_model_for_monomers or use FoldRun multimer."
    )

# Write validated FASTA file for foldrun-shim (CLI and Python Helper)
inputs_dir = Path("./inputs")
inputs_dir.mkdir(parents=True, exist_ok=True)
safe_stem = (
    "".join(c if c.isalnum() or c in ("_", "-") else "_" for c in target_name.strip()) or "target"
)
fasta_path = inputs_dir / f"{safe_stem}.fasta"

chain_ids = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
fasta_lines = []
for idx, seq in enumerate(sequences):
    chain_label = chain_ids[idx % len(chain_ids)]
    fasta_lines.append(f">{safe_stem}_chain_{chain_label} length={len(seq)}")
    fasta_lines.append(seq)

fasta_path.write_text("\n".join(fasta_lines) + "\n", encoding="utf-8")
recommended_gpu = (
    ("A100_80GB" if total_sequence_length >= 1000 else "A100")
    if model_type_to_use == ModelType.MULTIMER
    else ("A100_80GB" if total_sequence_length > 1500 else "A100")
)
print(
    f"Wrote FASTA ({len(sequences)} chain(s), {total_sequence_length} total residues) -> {fasta_path}"
)
print(f"FoldRun auto-selected GPU tier: {recommended_gpu}")

---
## Part B — FoldRun Additions (`!run_alphafold.py` CLI, Full 2.6 TB `/mnt/nfs/foldrun` Genetic Databases vs. DeepMind's Reduced Colab Databases, and Multi-Seed Vertex AI GPU Execution)

FoldRun upgrades AlphaFold 2 from a single-GPU reduced-database Colab notebook to a full-scale institutional prediction platform:
1. **Full 2.6 TB Genetic Databases on Filestore NFS (`/mnt/nfs/foldrun`)**: Mounts full `uniref90`, `mgnify`, `bfd` (metaclust clu complete), `uniref30`, `pdb70`, and full `pdb_mmcif` templates rather than DeepMind's small `small_bfd` reduced subset.
2. **Dual Interfaces**: Native drop-in CLI `!run_alphafold.py` with automatic host-flag sanitization (`_strip_local_alphafold2_args`) and programmatic Python Helper API (`alphafold2.predict(...)`).
3. **Multi-Seed Vertex AI GPU Fan-Out**: Predicts across multiple random seeds in parallel on NVIDIA A100 GPUs with DWS `FLEX_START`.
4. **Interrupt Resilience**: Persists `.foldrun_job.json` and uses server-side SHA-256 caching so re-running an interrupted cell reconnects instantly.


## 3. Configure FoldRun Cloud Databases (`/mnt/nfs/foldrun`) & MSA Search

### How FoldRun's 2.5 TB Cloud Filestore NFS Database Works Under the Hood
In the original DeepMind Colab, cell 4 had to download **300 separate HTTP chunks** (`uniref90`, `smallbfd`, `mgnify`, `uniprot`) into a 9 GB `tmpfs` `/tmp/ramdisk` and run single-node CPU `jackhmmer` against each chunk sequentially—and had to **disable structural templates entirely (`num_templates=0`)** and **forbid `full_dbs`** because the 2.5 TB database does not fit on a Colab VM.

With `foldrun-shim`, the `data_pipeline` Vertex AI component mounts FoldRun's pre-provisioned **2.5 TB `BASIC_SSD` Cloud Filestore NFS (`/mnt/nfs/foldrun`)** over low-latency private VPC networking (`databases.yaml`):

| Database on `/mnt/nfs/foldrun` | NFS Subpath | Shared Across Models | Used in `reduced_dbs` | Used in `full_dbs` | Search Tool |
| :--- | :--- | :--- | :---: | :---: | :--- |
| **UniRef90** | `uniref90/uniref90.fasta` (`+ uniref90_mmseqs`) | `af2`, `of3`, `boltz` | ✅ | ✅ | `jackhmmer` or GPU `mmseqs2` |
| **MGnify** | `mgnify/mgy_clusters_2022_05.fa` (`+ mgnify_mmseqs`) | `af2`, `of3`, `boltz` | ✅ | ✅ | `jackhmmer` or GPU `mmseqs2` |
| **Small BFD** (~17 GB) | `small_bfd/bfd-first_non_consensus_sequences.fasta` (`+ small_bfd_mmseqs`) | `af2` | ✅ (`use_small_bfd=True`) | ❌ | `jackhmmer` or GPU `mmseqs2` |
| **Full BFD** (~1.8 TB) | `bfd/bfd_metaclust_clu_complete_id30_c90_final_seq.sorted_opt` | `af2` | ❌ | ✅ (`use_small_bfd=False`) | `hhblits` (with UniRef30) |
| **UniRef30** (~206 GB) | `uniref30/UniRef30_2021_03` | `af2` | ❌ | ✅ (`use_small_bfd=False`) | `hhblits` (with Full BFD) |
| **PDB70** (Monomer Templates) | `pdb70/pdb70` | `af2` | ✅ | ✅ | `hhsearch` |
| **PDB SeqRes** (Multimer Templates) | `pdb_seqres/pdb_seqres.txt` | `af2`, `of3`, `boltz` | ✅ | ✅ | `hmmsearch` / `jackhmmer` |
| **PDB mmCIF + Obsolete** | `pdb_mmcif/mmcif_files` & `pdb_mmcif/obsolete.dat` | `af2`, `of3` | ✅ | ✅ | Coordinate extraction (`<= max_template_date`) |
| **UniProt** (Multimer Pairing) | `uniprot/uniprot.fasta` (TrEMBL + Swiss-Prot) | `af2`, `of3` | ✅ (Multimer) | ✅ (Multimer) | `jackhmmer` cross-chain species pairing |
| **AlphaFold 2 Weights** | `alphafold2/params` (`gs://<db-bucket>/alphafold2`) | `af2`, `bindcraft` | ✅ | ✅ | Loaded by 5 parallel A100 model runners |

### Choosing `--db_preset` and `--msa_method`
- **`--db_preset=reduced_dbs` (`use_small_bfd=True`)**:
  - Searches `uniref90`, `mgnify`, `small_bfd`, `pdb70`/`pdb_seqres` + `pdb_mmcif`, and `uniprot` (for multimers).
  - Supports **both** `--msa_method=jackhmmer` (CPU `c2-standard-16`) and **`--msa_method=mmseqs2`** (GPU-accelerated search on `g2-standard-12` with `NVIDIA_L4` against pre-built `uniref90_mmseqs`, `mgnify_mmseqs`, and `small_bfd_mmseqs` indexes on NFS).
- **`--db_preset=full_dbs` (`use_small_bfd=False`)**:
  - **Exclusive Cloud Advantage**: Searches the full 1.8 TB **BFD** + 206 GB **UniRef30** HMM profile databases using `hhblits` alongside `uniref90`, `mgnify`, `pdb70`/`pdb_seqres`, `pdb_mmcif`, and `uniprot`.
  - Requires `--msa_method=jackhmmer` or `--msa_method=auto` (since `full_dbs` uses HHsuite profile databases for BFD + UniRef30; passing `--msa_method=mmseqs2` with `full_dbs` is rejected with a clear validation message).

### Two-Tier Caching & Deduplication on FoldRun
1. **NFS Feature Cache (`/mnt/nfs/foldrun/af2_features_cache/<preset>_<sha256>/`)**:
   - Inside the `data_pipeline` KFP component (`foldrun_app/models/af2/pipeline/components/data_pipeline.py`), FoldRun hashes `{sequence, max_template_date, preset, use_small_bfd, msa_method}` with SHA-256.
   - If `/mnt/nfs/foldrun/af2_features_cache/<preset>_<sha256>/features.pkl` and `metadata.json` already exist on NFS, the data pipeline **completes in <1 second** by copying the cached `features.pkl` directly to the KFP output artifact—completely bypassing `jackhmmer`/`hhblits`/`mmseqs2` when you re-fold a sequence with different model presets, relaxation settings, or multimer seed counts!
2. **Server-Side Job Deduplication (`labels.input_sha256`) & Client `.foldrun_job.json`**:
   - `foldrun-gateway` computes a canonical `input_sha256` over the FASTA bytes and normalized CLI flags (`flag_allowlist.py`) and attaches `labels.input_sha256` to the Vertex AI `PipelineJob`.
   - Unless `--override` (`override=True`) is set, identical jobs automatically re-attach to the existing `PENDING`, `RUNNING`, or `SUCCEEDED` pipeline run.

In [ ]:
# @title 3. Configure FoldRun Database Preset, MSA Method & Scientific Flags
# @markdown Select the database preset (`full_dbs` vs `reduced_dbs`), MSA search method (`auto`, `jackhmmer`, `mmseqs2`), template cutoff date, and relaxation mode.
# @markdown All flags are validated against `foldrun_app/gateway/flag_allowlist.py` (`_validate_alphafold2_flags`).

db_preset = "full_dbs"  # @param ["full_dbs", "reduced_dbs"]
msa_method = "auto"  # @param ["auto", "jackhmmer", "mmseqs2"]
max_template_date = "2025-04-01"  # @param {type:"string"}
models_to_relax = "best"  # @param ["best", "all", "none"]
monomer_model_preset = "monomer_ptm"  # @param ["monomer", "monomer_ptm", "monomer_casp14"]
num_multimer_predictions_per_model = 5  # @param {type:"slider", min:1, max:25, step:1}
output_dir = "./af2_results"  # @param {type:"string"}
override_existing_job = False  # @param {type:"boolean"}

# Resolve model_preset based on sequence count / multimer selection
if model_type_to_use == ModelType.MULTIMER:
    model_preset = "multimer"
else:
    model_preset = monomer_model_preset

# Validate msa_method + db_preset combination upfront
if msa_method == "mmseqs2" and db_preset == "full_dbs":
    print(
        "NOTE: --msa_method=mmseqs2 requires --db_preset=reduced_dbs (use_small_bfd=True) "
        "because full BFD + UniRef30 use HHblits profile databases. Switching db_preset -> 'reduced_dbs'."
    )
    db_preset = "reduced_dbs"

print("Configured AlphaFold 2 FoldRun Parameters:")
print(f"  --fasta_paths                        = {fasta_path}")
print(f"  --output_dir                         = {output_dir}")
print(f"  --model_preset                       = {model_preset}")
print(
    f"  --db_preset                          = {db_preset} (use_small_bfd={db_preset == 'reduced_dbs'})"
)
print(f"  --msa_method                         = {msa_method}")
print(f"  --max_template_date                  = {max_template_date}")
print(f"  --models_to_relax                    = {models_to_relax}")
if model_preset == "multimer":
    print(
        f"  --num_multimer_predictions_per_model = {num_multimer_predictions_per_model} ({num_multimer_predictions_per_model * 5} total structures)"
    )

## 4. Run AlphaFold 2 on FoldRun (Both Shim Interfaces)

`foldrun-shim` provides **two interchangeable interfaces** that both submit to `POST /v1/alphafold2/jobs` on `foldrun-gateway`, persist `<output_dir>/.foldrun_job.json` for interrupt/disconnect recovery, poll `GET /v1/jobs/{job_id}` with live `tqdm` step progress (`data-pipeline`, `model-predict-1..5`, `relax`), and extract `GET /v1/jobs/{job_id}/outputs` into `<output_dir>/<fasta_stem>/`:

### Interface A: Python Notebook Helper API (`alphafold2.predict`)
Ideal for interactive Jupyter/Colab notebooks. Returns a `GatewayJobResult` object whose `_repr_html_()` method renders a styled HTML card with:
- **Job ID** (`alphafold2-gw-<sha4>-<nonce>-<YYYYMMDDHHMMSS>`)
- **Pipeline State** (`PIPELINE_STATE_SUCCEEDED`)
- **Materialized Local Output Directory** (`./af2_results/<fasta_stem>/...`)
- **Direct Link to the 3D FoldRun Viewer** (`job.viewer_url`)

### Interface B: Drop-In Native CLI (`!run_alphafold.py ...`)
Ideal for running unmodified shell commands copied from DeepMind's AlphaFold README or existing bioinformatics scripts.
- `_strip_local_alphafold2_args()` in `foldrun_shim/cli.py` **automatically strips host-specific local database and binary flags** (`--data_dir`, `--*_database_path`, `--*_binary_path`, `--pdb70_database_path`, `--template_mmcif_dir`, `--obsolete_pdbs_path`, `--use_gpu_relax`, `--use_precomputed_msas`) so commands written for local workstations work without editing!
- All scientific flags (`--model_preset`, `--db_preset`, `--max_template_date`, `--models_to_relax`, `--num_multimer_predictions_per_model`, `--msa_method`, `--benchmark`) are validated by `foldrun-gateway` (`_validate_alphafold2_flags` in `flag_allowlist.py`) and forwarded to the Vertex AI A100 pipeline.

> [!IMPORTANT]
> **Disconnect & Kernel Restart Resilience (`.foldrun_job.json`)**
> As soon as `foldrun-gateway` returns `202 Accepted`, `foldrun-shim` writes `<output_dir>/.foldrun_job.json` containing `job_id`, `model`, `input_sha256`, `server_input_sha256`, and `viewer_url`.
> If your Colab tab disconnects, your laptop sleeps, or you run Interface B right after Interface A with the same flags, `foldrun-shim` detects the matching `input_sha256` in `.foldrun_job.json`, verifies the job on `foldrun-gateway`, and **re-attaches immediately** without launching a duplicate A100 GPU run!

In [ ]:
# @title 4A. Interface 1 — Python Notebook Helper API (`alphafold2.predict`)
# @markdown Executes AlphaFold 2 on FoldRun's Vertex AI pipeline via `alphafold2.predict(...)` and renders the interactive `GatewayJobResult` HTML status card.
# @markdown Hardware & scheduling options below default to `inherit`, reading `FOLDRUN_GPU_TYPE` and `FOLDRUN_ENABLE_FLEX_START` from the environment (or Cell 1B).

gpu_type_selection = "inherit from environment / auto"  # @param ["inherit from environment / auto", "L4", "A100", "A100_80GB"]
scheduling_selection = "inherit from environment"  # @param ["inherit from environment", "DWS FLEX_START (discounted)", "STANDARD On-Demand (immediate)"]

gpu_type_override = None if gpu_type_selection.startswith("inherit") else gpu_type_selection
if scheduling_selection == "DWS FLEX_START (discounted)":
    enable_flex_start_override = True
elif scheduling_selection == "STANDARD On-Demand (immediate)":
    enable_flex_start_override = False
else:
    enable_flex_start_override = None

import json

from IPython.display import display

predict_kwargs = {
    "db_preset": db_preset,
    "models_to_relax": models_to_relax,
    "msa_method": msa_method,
}
if model_preset == "multimer":
    predict_kwargs["num_multimer_predictions_per_model"] = num_multimer_predictions_per_model

job = alphafold2.predict(
    fasta_paths=fasta_path,
    output_dir=output_dir,
    model_preset=model_preset,
    max_template_date=max_template_date,
    override=override_existing_job,
    gpu_type=gpu_type_override,
    enable_flex_start=enable_flex_start_override,
    **predict_kwargs,
)

# Inspect the disconnect-recovery state file written by foldrun-shim
state_file = Path(output_dir) / ".foldrun_job.json"
if state_file.exists():
    print("Saved interrupt-recovery state (.foldrun_job.json):")
    print(json.dumps(json.loads(state_file.read_text(encoding="utf-8")), indent=2))

# Display the rich HTML status card (with 3D FoldRun Viewer link)
display(job)

In [ ]:
# Execute the drop-in native CLI command via shell magic (automatically deduplicates against Cell 4A!)
!run_alphafold.py --fasta_paths={fasta_path} --output_dir={output_dir} --model_preset={model_preset} --db_preset={db_preset} --max_template_date={max_template_date} --models_to_relax={models_to_relax} --msa_method={msa_method} --data_dir=/mnt/nfs/foldrun --uniref90_database_path=/mnt/nfs/foldrun/uniref90/uniref90.fasta --pdb70_database_path=/mnt/nfs/foldrun/pdb70/pdb70 --template_mmcif_dir=/mnt/nfs/foldrun/pdb_mmcif/mmcif_files --use_gpu_relax=true

## 5. Inspect FoldRun NFS Database Outputs (`msas/`, `features.pkl`, `ranking_debug.json`)

When the Vertex AI `PipelineJob` completes, `foldrun-gateway` (`_package_alphafold2_outputs` in `output_packager.py`) streams the GCS artifacts directly into `native_outputs.tar.gz` and returns a 15-minute V4 Signed URL. `foldrun-shim` unpacks the archive into `<output_dir>/<fasta_stem>/`, reconstructing the exact local AlphaFold 2 output layout:

```text
<output_dir>/<fasta_stem>/
├── ranking_debug.json          # {"plddts": {"model_1_ptm": 92.4, ...}, "order": ["model_1_ptm", ...]}
├── ranked_0.pdb .. ranked_4.pdb # Models sorted by ranking_confidence descending (pLDDT or 0.8*ipTM + 0.2*pTM)
├── relaxed_model_*.pdb         # AMBER-relaxed PDB(s) (top-1 when models_to_relax=best, all when models_to_relax=all)
├── unrelaxed_model_*.pdb       # Raw unrelaxed structures from each parallel A100 model task
├── features.pkl                # NumPy feature dict (sequence, MSA alignments, PDB70/mmCIF template coordinates)
└── msas/                       # Alignments produced against /mnt/nfs/foldrun databases:
    ├── uniref90_hits.sto       # JackHMMER / MMseqs2 hits against /mnt/nfs/foldrun/uniref90
    ├── mgnify_hits.sto         # JackHMMER / MMseqs2 hits against /mnt/nfs/foldrun/mgnify
    ├── bfd_uniref_hits.a3m     # HHblits hits against /mnt/nfs/foldrun/bfd + uniref30 (when db_preset=full_dbs)
    ├── small_bfd_hits.sto      # JackHMMER / MMseqs2 hits against /mnt/nfs/foldrun/small_bfd (when db_preset=reduced_dbs)
    ├── pdb_hits.hhr / .sto     # HHsearch (pdb70) or HMMsearch (pdb_seqres) structural template hits
    └── uniprot_hits.sto        # Cross-chain species pairing hits against /mnt/nfs/foldrun/uniprot (multimer)
```

Execute the cell below to inspect the alignments, structural template hits, model rankings, and per-residue MSA coverage returned from FoldRun's `/mnt/nfs/foldrun` database search.

In [ ]:
# @title 5. Inspect `msas/`, `features.pkl`, and `ranking_debug.json` from `/mnt/nfs/foldrun`
# @markdown Displays the model ranking table, lists the MSA & template hit files generated against FoldRun's 2.5 TB Cloud Filestore NFS share, and plots per-residue MSA depth (matching `notebook_utils.show_msa_info` from the original DeepMind Colab).

import json
import pickle

import matplotlib.pyplot as plt
import numpy as np

target_out_dir = Path(output_dir) / fasta_path.stem
print(f"Inspecting materialized AlphaFold 2 output directory: {target_out_dir.resolve()}\n")

# 1. Inspect ranking_debug.json
ranking_path = target_out_dir / "ranking_debug.json"
if ranking_path.exists():
    ranking_data = json.loads(ranking_path.read_text(encoding="utf-8"))
    order = ranking_data.get("order", [])
    scores = ranking_data.get("plddts", {})
    metric_label = "0.8*ipTM + 0.2*pTM" if model_preset == "multimer" else "Mean pLDDT"
    print(f"=== Model Ranking ({metric_label}) ===")
    for rank_idx, mname in enumerate(order):
        score = scores.get(mname, 0.0)
        print(f"  ranked_{rank_idx}.pdb  <-  {mname:<25} ({metric_label}: {score:.2f})")
    print()

# 2. Inspect msas/ directory returned from /mnt/nfs/foldrun
msas_dir = target_out_dir / "msas"
if msas_dir.exists():
    msa_files = sorted(p for p in msas_dir.rglob("*") if p.is_file())
    print(f"=== Genetic & Template Hit Files in {msas_dir} ({len(msa_files)} files) ===")
    for msa_file in msa_files:
        rel_name = msa_file.relative_to(msas_dir)
        size_kb = msa_file.stat().st_size / 1024.0
        # Count alignment sequences in .sto or .a3m files
        seq_count_str = ""
        try:
            text_head = msa_file.read_text(encoding="utf-8", errors="ignore")
            if msa_file.suffix == ".a3m":
                n_seqs = sum(1 for line in text_head.splitlines() if line.startswith(">"))
                seq_count_str = f" | {n_seqs:,} aligned sequences"
            elif msa_file.suffix == ".sto":
                n_seqs = sum(
                    1
                    for line in text_head.splitlines()
                    if line and not line.startswith("#") and not line.startswith("//")
                )
                seq_count_str = f" | {n_seqs:,} alignment rows"
        except Exception:
            pass
        print(f"  - msas/{rel_name} ({size_kb:,.1f} KB{seq_count_str})")
    print()

# 3. Inspect features.pkl & plot Per-Residue Non-Gap MSA Coverage (notebook_utils.show_msa_info)
features_path = target_out_dir / "features.pkl"
if features_path.exists():
    with open(features_path, "rb") as f:
        feature_dict = pickle.load(f)

    msa_matrix = feature_dict.get("msa")
    num_alignments_arr = feature_dict.get("num_alignments")
    num_templates = 0
    if "template_domain_names" in feature_dict:
        num_templates = len(feature_dict["template_domain_names"])
    elif "template_aatype" in feature_dict:
        num_templates = feature_dict["template_aatype"].shape[0]

    dedup_msa_size = (
        int(np.asarray(num_alignments_arr).flat[0])
        if num_alignments_arr is not None
        else (int(msa_matrix.shape[0]) if msa_matrix is not None else 0)
    )
    print("=== NFS Feature Summary (features.pkl) ===")
    print(f"  Final deduplicated MSA depth : {dedup_msa_size:,} sequences")
    print(
        f"  Structural templates matched : {num_templates} PDB mmCIF templates (cutoff <= {max_template_date})"
    )
    print(f"  Feature keys materialized    : {sorted(feature_dict.keys())}\n")

    if msa_matrix is not None and msa_matrix.ndim == 2:
        # In AlphaFold's HHblits/residue_constants encoding, gap '-' is index 21
        GAP_INDEX = 21
        non_gap_count = np.sum(msa_matrix != GAP_INDEX, axis=0)
        total_msa_rows = msa_matrix.shape[0]

        plt.figure(figsize=(12, 3.2))
        plt.title(f"Per-Residue Count of Non-Gap Amino Acids in the MSA ({fasta_path.stem})")
        plt.plot(non_gap_count, color="black", linewidth=1.5)
        plt.fill_between(range(len(non_gap_count)), non_gap_count, color="#65CBF3", alpha=0.25)
        plt.xlabel("Residue Index")
        plt.ylabel("Non-Gap Count")
        plt.ylim(0, max(1, total_msa_rows))
        plt.grid(axis="y", linestyle="--", alpha=0.4)
        plt.tight_layout()
        plt.show()

## 6. Visualize 3D Structure (`py3Dmol`) & Confidence Plots (pLDDT & PAE)

Just like Cell 5 of the original DeepMind `AlphaFold.ipynb` notebook, the cell below renders:
1. **Interactive 3D Structure (`py3Dmol`)**:
   - Reads `<output_dir>/<fasta_stem>/ranked_0.pdb` (the top-ranked AMBER-relaxed structure).
   - Colors each residue by its AlphaFold **pLDDT confidence band** (stored in the PDB B-factor column):
     - <span style="color:#0053D6;font-weight:bold;">■ Very high ($\text{pLDDT} > 90$)</span>: `#0053D6`
     - <span style="color:#65CBF3;font-weight:bold;">■ Confident ($90 \ge \text{pLDDT} > 70$)</span>: `#65CBF3`
     - <span style="color:#FFDB13;font-weight:bold;">■ Low ($70 \ge \text{pLDDT} > 50$)</span>: `#FFDB13`
     - <span style="color:#FF7D45;font-weight:bold;">■ Very low ($\text{pLDDT} \le 50$)</span>: `#FF7D45`
   - For **Multimers**, also renders a second 3D viewer colored by **Chain ID** (`A`, `B`, `C`, ...) to highlight inter-chain interfaces.
2. **Per-Residue pLDDT & Predicted Aligned Error (PAE) Plots**:
   - Plots per-residue pLDDT across all 5 ranked models (`ranked_0.pdb` .. `ranked_4.pdb`).
   - Plots the 2D **Predicted Aligned Error (PAE)** matrix (with red dashed chain-boundary lines for multimers) if `result_model_*.pkl` or `predicted_aligned_error.json` is present.
3. **Cloud-Hosted 3D FoldRun Viewer (`job.viewer_url`)**:
   - In addition to inline `py3Dmol` rendering, `foldrun-gateway` automatically triggers `foldrun-analysis-job` when the pipeline succeeds and provides `job.viewer_url` for full-screen interactive 3D structure & PAE exploration in your browser.

In [ ]:
# @title 6. Render 3D Structure in `py3Dmol` & Plot pLDDT / PAE Confidence
# @markdown Visualizes `ranked_0.pdb` colored by AlphaFold pLDDT confidence bands (and by chain for multimers) and plots per-residue pLDDT and PAE.

import json
import pickle
import shutil

import matplotlib.pyplot as plt
import numpy as np
import py3Dmol

PLDDT_BANDS = [
    (0, 50, "#FF7D45", "Very low (pLDDT < 50)"),
    (50, 70, "#FFDB13", "Low (70 > pLDDT > 50)"),
    (70, 90, "#65CBF3", "Confident (90 > pLDDT > 70)"),
    (90, 100, "#0053D6", "Very high (pLDDT > 90)"),
]

show_sidechains = True  # @param {type:"boolean"}
create_zip_archive = False  # @param {type:"boolean"}


def parse_pdb_plddt_and_chains(pdb_path: Path) -> tuple[list[float], list[str], str]:
    """Extract per-residue pLDDT (from CA atom B-factors) and rewrite banded B-factors for py3Dmol."""
    raw_lines = pdb_path.read_text(encoding="utf-8").splitlines()
    ca_plddts: list[float] = []
    ca_chains: list[str] = []
    seen_res: set[tuple[str, str]] = set()
    banded_lines: list[str] = []

    for line in raw_lines:
        if line.startswith(("ATOM  ", "HETATM")) and len(line) >= 66:
            atom_name = line[12:16].strip()
            chain_id = line[21:22].strip() or "A"
            res_seq = line[22:27].strip()
            try:
                bfac = float(line[60:66].strip())
            except ValueError:
                bfac = 0.0

            if atom_name == "CA" and (chain_id, res_seq) not in seen_res:
                seen_res.add((chain_id, res_seq))
                ca_plddts.append(bfac)
                ca_chains.append(chain_id)

            # Map pLDDT (0..100) to band index (0..3) in B-factor column for exact DeepMind Colab coloring
            band_idx = 0
            for idx, (low, high, _, _) in enumerate(PLDDT_BANDS):
                if low <= bfac <= high:
                    band_idx = idx
                    break
            banded_lines.append(f"{line[:60]}{band_idx:6.2f}{line[66:]}")
        else:
            banded_lines.append(line)

    return ca_plddts, ca_chains, "\n".join(banded_lines) + "\n"


ranked_0_path = target_out_dir / "ranked_0.pdb"
if not ranked_0_path.exists():
    raise FileNotFoundError(
        f"Expected top-ranked prediction at {ranked_0_path}. Run Cell 4A or 4B first."
    )

plddt_vals, chain_ids_per_res, to_visualize_pdb = parse_pdb_plddt_and_chains(ranked_0_path)
unique_chains = sorted(set(chain_ids_per_res))
print(
    f"Loaded {ranked_0_path.name}: {len(plddt_vals)} residues across {len(unique_chains)} chain(s) "
    f"| Mean pLDDT = {np.mean(plddt_vals):.2f}"
)
if getattr(job, "viewer_url", None):
    print(f"Cloud 3D FoldRun Viewer URL: {job.viewer_url}")

# 1. Show structure colored by chain if multimer
if len(unique_chains) > 1 or model_type_to_use == ModelType.MULTIMER:
    print("\n3D Structure Colored by Chain (Multimer View):")
    multichain_view = py3Dmol.view(width=800, height=500)
    multichain_view.addModelsAsFrames(to_visualize_pdb)
    multichain_view.setStyle({"model": -1}, {"cartoon": {"colorscheme": "chain"}})
    multichain_view.zoomTo()
    multichain_view.show()

# 2. Show structure colored by per-residue pLDDT confidence band
print("\n3D Structure Colored by Per-Residue pLDDT Confidence:")
color_map = {i: band[2] for i, band in enumerate(PLDDT_BANDS)}
view = py3Dmol.view(width=800, height=500)
view.addModelsAsFrames(to_visualize_pdb)
style: dict = {"cartoon": {"colorscheme": {"prop": "b", "map": color_map}}}
if show_sidechains:
    style["stick"] = {}
view.setStyle({"model": -1}, style)
view.zoomTo()
view.show()

# 3. Check for PAE data (from result_model_*.pkl or predicted_aligned_error.json)
pae_matrix = None
max_pae = 31.75
for pkl_candidate in sorted(target_out_dir.glob("result_model_*.pkl")):
    try:
        with open(pkl_candidate, "rb") as pf:
            res_dict = pickle.load(pf)
        if "predicted_aligned_error" in res_dict:
            pae_matrix = np.asarray(res_dict["predicted_aligned_error"])
            max_pae = float(res_dict.get("max_predicted_aligned_error", 31.75))
            break
    except Exception:
        pass

pae_json_path = target_out_dir / "predicted_aligned_error.json"
if pae_matrix is None and pae_json_path.exists():
    pae_loaded = json.loads(pae_json_path.read_text(encoding="utf-8"))
    if isinstance(pae_loaded, list) and pae_loaded and "predicted_aligned_error" in pae_loaded[0]:
        pae_matrix = np.asarray(pae_loaded[0]["predicted_aligned_error"])
        max_pae = float(pae_loaded[0].get("max_predicted_aligned_error", 31.75))

# 4. Plot pLDDT across all ranked models + PAE (if available)
num_plots = 2 if pae_matrix is not None else 1
plt.figure(figsize=(7.5 * num_plots, 5.2))

plt.subplot(1, num_plots, 1)
for rank_idx in range(5):
    cand_pdb = target_out_dir / f"ranked_{rank_idx}.pdb"
    if cand_pdb.exists():
        cand_plddt, _, _ = parse_pdb_plddt_and_chains(cand_pdb)
        alpha = 1.0 if rank_idx == 0 else 0.45
        lw = 2.0 if rank_idx == 0 else 1.0
        plt.plot(
            cand_plddt,
            label=f"ranked_{rank_idx} (mean={np.mean(cand_plddt):.1f})",
            alpha=alpha,
            linewidth=lw,
        )

for low, high, color, _label in PLDDT_BANDS:
    plt.axhspan(low, high, color=color, alpha=0.12)

plt.title("Predicted LDDT (pLDDT) per Residue")
plt.xlabel("Residue")
plt.ylabel("pLDDT")
plt.ylim(0, 100)
plt.legend(loc="lower right", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.3)

if num_plots == 2 and pae_matrix is not None:
    plt.subplot(1, 2, 2)
    plt.imshow(pae_matrix, vmin=0.0, vmax=max_pae, cmap="Greens_r")
    plt.colorbar(fraction=0.046, pad=0.04, label="Expected Position Error (Å)")
    total_res = len(chain_ids_per_res)
    for idx in range(1, total_res):
        if chain_ids_per_res[idx] != chain_ids_per_res[idx - 1]:
            plt.axhline(idx - 0.5, color="red", linewidth=1.2)
            plt.axvline(idx - 0.5, color="red", linewidth=1.2)
    plt.title("Predicted Aligned Error (PAE)")
    plt.xlabel("Scored residue")
    plt.ylabel("Aligned residue")

plt.tight_layout()
plt.show()

if create_zip_archive:
    zip_path = shutil.make_archive(
        base_name=str(target_out_dir), format="zip", root_dir=str(target_out_dir)
    )
    print(f"Created zip archive of predictions: {zip_path}")

## 7. Interpreting Results, Citations & License

### Interpreting AlphaFold 2 Predictions
- **Predicted LDDT (pLDDT)** measures local intra-domain confidence on a $0\text{–}100$ scale ($\text{pLDDT} > 90$ indicates high accuracy suitable for characterizing binding sites; $70 < \text{pLDDT} \le 90$ indicates good backbone accuracy; $\text{pLDDT} < 50$ often corresponds to intrinsically disordered regions).
- **Predicted Aligned Error (PAE)** measures confidence in the relative packing of domains or chains (lower values in dark green indicate rigid inter-domain or inter-chain orientation).

### Citing This Work
If you disclose findings arising from this notebook, please cite:
- **AlphaFold 2**: Jumper, J. *et al.* Highly accurate protein structure prediction with AlphaFold. *Nature* **596**, 583–589 (2021). https://doi.org/10.1038/s41586-021-03819-2
- **AlphaFold-Multimer**: Evans, R. *et al.* Protein complex prediction with AlphaFold-Multimer. *bioRxiv* (2021). https://doi.org/10.1101/2021.10.04.463034v1

### License
- **Notebook & `foldrun-shim` Code**: Licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).
- **AlphaFold Model Parameters**: Subject to the Creative Commons Attribution 4.0 International ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/legalcode)) license.